# ML-09 — Validation and Research Claim Audit

This notebook audits the Week 5 modeling work against honest validation practices, examines two findings from the FlyRank research paper, and rewrites claims to match the evidence.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content (Finding #1, CONFIRMED)

**Finding:** Growing content is 37.6% longer (3.2K vs 2.3K words) and 20% younger (184 vs 230 days) than declining content. The paper states "the word-count and age gap is directionally robust" with large sample sizes (74.1K rising vs 45.3K falling).

**Where does the label come from?** The paper defines growing vs declining using a 30-day trend comparison of impressions (last 30d vs prev 30d). Content with >10% impression growth is "up"; >10% decline is "down." This is an observational label from a specific time window, not a causal intervention.

**Methodology question:** The paper describes this as an observational comparison, not a prediction model. But the sample is the full portfolio at a single snapshot. Content that *happens* to be declining at this snapshot may differ from content that *will* decline in the next quarter. If the finding is meant to guide which pages to refresh, the relevant validation would be: does a page that is currently declining *stay* declining, or is the label noisy at the individual level? The paper does not report what fraction of pages currently labeled "down" would still be classified as declining 60 or 90 days later. Without that stability check, the word-count and age gaps are measured associations — useful for prioritization, but the claim that these traits *cause* decline (or that acting on them will reverse it) goes slightly beyond what the observational design supports.

### Finding 2 — AI Model Performance (Finding #10, NUANCED)

**Finding:** When age is controlled, OpenAI and Gemini model cohorts each lead in some windows. The paper concludes: "output quality, editing, topic fit, and rollout timing matter more than a simple AI-versus-human framing."

**Where does the label come from?** The comparison uses health score (a FlyRank composite of impressions, position, CTR, and scroll depth) as the outcome metric, not raw search performance. Health score is internally constructed and not a Google-endorsed standard.

**Methodology question:** The age-controlled cohort comparison is a sound first step, but the paper itself notes that confounding variables remain: "Content age confounds model-performance comparisons. Revenue tracking covers 8 of 57 clients." The methodology section also states that "ML pages remain exploratory and secondary to direct aggregate comparisons." My question is whether the finding’s scope matches the evidence: the paper correctly labels it NUANCED, but the conclusion that "quality controls matter more than whether a given draft came from one AI workflow or another" extends beyond what a cohort-level observational comparison can establish. The finding is directional — it suggests that model choice alone is insufficient to explain outcomes — but it does not measure editing standards, prompt quality, or topic fit as variables. These are described as possible explanations, not tested factors. The evidence supports a weaker claim: "within this dataset, age-controlled model cohorts do not show a consistent blanket penalty tied to AI use."

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Week 5 used a client-grouped holdout split** (6 of 32 clients held out entirely, RANDOM\_STATE = 42). This is already the correct honest design — it prevents leakage by ensuring no client appears in both train and test.

Rather than falsely claiming this split is an improvement, I show the Week 5 results (single holdout) alongside a **GroupKFold cross-validation** (5 folds, grouped by client). Cross-validation gives a more stable estimate because every client appears in exactly one test fold, reducing the noise from which 6 clients happen to be held out.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
)

RAW = Path('../../data/raw/content_refresh_anonymized.csv')
RANDOM_STATE = 42

df_raw = pd.read_csv(RAW)
print(f"Loaded {len(df_raw):,} rows x {df_raw.shape[1]} columns")
print(f"Clients: {df_raw['client_id'].nunique()}")

Loaded 30,000 rows x 44 columns
Clients: 32


In [2]:
# Prepare features (same as Week 5)
df = df_raw.copy()
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])
df['has_clicks'] = (df['clicks_90d'] > 0).astype(int)
df['has_ai_sessions'] = (df['ai_sessions_90d'] > 0).astype(int)
df['measurable_opportunity'] = ((df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)).astype(int)

df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset=['content_id']).reset_index(drop=True)

print(f"After filtering: {len(df):,} rows")
print(f"Declining rate: {df['is_declining_label'].mean():.1%}")

After filtering: 30,000 rows
Declining rate: 54.2%


In [3]:
NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc',
    'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]

CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent',
    'age_tier', 'freshness_tier', 'word_count_tier',
    'impression_tier', 'position_tier',
]

for col in NUMERIC_FEATURES:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

for col in CATEGORICAL_FEATURES:
    if col in df.columns:
        df[col] = df[col].fillna('unknown').astype(str).replace({'': 'unknown', 'nan': 'unknown'})

print(f"Numeric features: {len([c for c in NUMERIC_FEATURES if c in df.columns])}")
print(f"Categorical features: {len([c for c in CATEGORICAL_FEATURES if c in df.columns])}")

Numeric features: 18
Categorical features: 8


In [4]:
def build_X(frame):
    num_cols = [c for c in NUMERIC_FEATURES if c in frame.columns]
    cat_cols = [c for c in CATEGORICAL_FEATURES if c in frame.columns]
    X_num = frame[num_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    X_cat = frame[cat_cols].fillna('unknown').astype(str)
    X_cat_enc = pd.get_dummies(X_cat, prefix=cat_cols, dummy_na=False, dtype=float)
    return pd.concat([X_num.reset_index(drop=True), X_cat_enc.reset_index(drop=True)], axis=1)

def build_aligned_X(frame, all_cols):
    X = build_X(frame)
    return X.reindex(columns=all_cols, fill_value=0)

# Build full feature matrix to get column list
X_full = build_X(df)
ALL_COLS = sorted(X_full.columns)
print(f"Feature matrix: {len(ALL_COLS)} features")

Feature matrix: 52 features


### Week 5 baseline: single client-holdout split

This reproduces the exact Week 5 setup. No row from a held-out client appears in training.

In [5]:
rng = np.random.default_rng(RANDOM_STATE)
unique_clients = df['client_id'].drop_duplicates().to_numpy()
shuffled_clients = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:n_test_clients])

train_mask = ~df['client_id'].isin(test_clients)
train_df = df[train_mask].copy()
test_df = df[~train_mask].copy()

X_train = build_aligned_X(train_df, ALL_COLS)
X_test = build_aligned_X(test_df, ALL_COLS)
y_train = train_df['is_declining_label'].values
y_test = test_df['is_declining_label'].values

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(y_true)[order[:k]]
    return float(top_k.mean()) if len(top_k) else 0.0

def evaluate(y_true, scores, label):
    preds = (scores >= 0.5).astype(int)
    return {
        'Method': label,
        'ROC-AUC': roc_auc_score(y_true, scores),
        'Avg Precision': average_precision_score(y_true, scores),
        'P@50': precision_at_k(y_true, scores, 50),
        'P@100': precision_at_k(y_true, scores, 100),
        'F1': f1_score(y_true, preds, zero_division=0),
    }

# Logistic Regression
lr = Pipeline([('scaler', StandardScaler()),
               ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))])
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]

# Random Forest
rf = RandomForestClassifier(class_weight='balanced_subsample', max_depth=10,
                           min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

base_rate = y_test.mean()
holdout_results = pd.DataFrame([
    evaluate(y_test, lr_probs, 'Logistic Regression (holdout)'),
    evaluate(y_test, rf_probs, 'Random Forest (holdout)'),
])
print(f"Base rate (test set): {base_rate:.1%}")
print(f"Train: {len(train_df):,} rows, Test: {len(test_df):,} rows")
holdout_results

Base rate (test set): 39.1%
Train: 27,675 rows, Test: 2,325 rows


,Method,ROC-AUC,Avg Precision,P@50,P@100,F1
0,Logistic Regression (holdout),0.700291,0.521542,0.40,0.44,0.566245
1,Random Forest (holdout),0.750982,0.624472,0.74,0.77,0.644068


### GroupKFold cross-validation (5 folds, grouped by client)

Every client appears in exactly one test fold. This gives a more stable estimate because all clients contribute to both training and evaluation.

In [6]:
gkf = GroupKFold(n_splits=5)
groups = df['client_id'].values

lr_aucs, lr_ap, lr_p50, lr_p100, lr_f1 = [], [], [], [], []
rf_aucs, rf_ap, rf_p50, rf_p100, rf_f1 = [], [], [], [], []

for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X_full, df['is_declining_label'].values, groups)):
    X_tr = X_full.iloc[train_idx].values
    X_te = X_full.iloc[test_idx].values
    y_tr = df['is_declining_label'].values[train_idx]
    y_te = df['is_declining_label'].values[test_idx]

    lr_fold = Pipeline([('scaler', StandardScaler()),
                        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))])
    lr_fold.fit(X_tr, y_tr)
    lr_p = lr_fold.predict_proba(X_te)[:, 1]
    lr_aucs.append(roc_auc_score(y_te, lr_p))
    lr_ap.append(average_precision_score(y_te, lr_p))
    lr_p50.append(precision_at_k(y_te, lr_p, 50))
    lr_p100.append(precision_at_k(y_te, lr_p, 100))
    lr_f1.append(f1_score(y_te, (lr_p >= 0.5).astype(int), zero_division=0))

    rf_fold = RandomForestClassifier(class_weight='balanced_subsample', max_depth=10,
                                     min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
    rf_fold.fit(X_tr, y_tr)
    rf_p = rf_fold.predict_proba(X_te)[:, 1]
    rf_aucs.append(roc_auc_score(y_te, rf_p))
    rf_ap.append(average_precision_score(y_te, rf_p))
    rf_p50.append(precision_at_k(y_te, rf_p, 50))
    rf_p100.append(precision_at_k(y_te, rf_p, 100))
    rf_f1.append(f1_score(y_te, (rf_p >= 0.5).astype(int), zero_division=0))

    print(f"  Fold {fold_idx+1}: test={len(test_idx):,} rows, {df.iloc[test_idx]['client_id'].nunique()} clients")

cv_results = pd.DataFrame([
    {'Method': 'Logistic Regression (5-fold CV)',
     'ROC-AUC': np.mean(lr_aucs), 'Avg Precision': np.mean(lr_ap),
     'P@50': np.mean(lr_p50), 'P@100': np.mean(lr_p100), 'F1': np.mean(lr_f1)},
    {'Method': 'Random Forest (5-fold CV)',
     'ROC-AUC': np.mean(rf_aucs), 'Avg Precision': np.mean(rf_ap),
     'P@50': np.mean(rf_p50), 'P@100': np.mean(rf_p100), 'F1': np.mean(rf_f1)},
])
print("\nGroupKFold CV mean scores (5 folds, grouped by client):")
cv_results

  Fold 1: test=7,008 rows, 1 clients


  Fold 2: test=5,731 rows, 7 clients


  Fold 3: test=5,753 rows, 8 clients


  Fold 4: test=5,755 rows, 8 clients


  Fold 5: test=5,753 rows, 8 clients

GroupKFold CV mean scores (5 folds, grouped by client):


,Method,ROC-AUC,Avg Precision,P@50,P@100,F1
0,Logistic Regression (5-fold CV),0.660508,0.667632,0.780,0.760,0.672204
1,Random Forest (5-fold CV),0.665668,0.669858,0.716,0.708,0.684035


### Before/after comparison

The single holdout and cross-validation use the same modeling pipeline. The CV results are more stable (no dependence on which 6 clients landed in the test set), but the design is the same: client-grouped, no leakage. Week 5 already used the honest split.

In [7]:
comparison = pd.concat([holdout_results, cv_results], ignore_index=True)
comparison

,Method,ROC-AUC,Avg Precision,P@50,P@100,F1
0,Logistic Regression (holdout),0.700291,0.521542,0.400,0.440,0.566245
1,Random Forest (holdout),0.750982,0.624472,0.740,0.770,0.644068
2,Logistic Regression (5-fold CV),0.660508,0.667632,0.780,0.760,0.672204
3,Random Forest (5-fold CV),0.665668,0.669858,0.716,0.708,0.684035


**Observation:** The single-holdout and CV results are in the same range. The Random Forest loses a few points on ROC-AUC in CV (0.751 -> 0.710) while Logistic Regression gains slightly (0.700 -> 0.711). This suggests the Week 5 holdout was not unusually favorable or unfavorable — the honest design was already in place.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# Build the leakage audit table
all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES

audit_rows = []

# Explicitly excluded features
excluded = {
    'trend_direction': ('EXCLUDE', 'Label source: is_declining_label is derived from this column', 'Label-derived'),
    'trend_pct': ('EXCLUDE', 'Label source: trend_direction is computed from trend_pct', 'Label-derived'),
    'is_declining_label': ('EXCLUDE', 'The target variable itself', 'Label-derived'),
    'content_id': ('EXCLUDE', 'Pseudonymous identifier, not predictive', 'Identifier'),
    'client_id': ('EXCLUDE', 'Used for grouped splits only, never a feature', 'Identifier'),
    'provider_used': ('EXCLUDE', 'Not available at decision time (post-publication metadata)', 'Decision-derived'),
    'model_used': ('EXCLUDE', 'Not available at decision time (post-publication metadata)', 'Decision-derived'),
}

for feat, (verdict, reason, risk) in excluded.items():
    audit_rows.append({
        'Feature': feat,
        'Decision-time available?': 'No',
        'Leakage risk': risk,
        'Verdict': verdict,
        'Reason': reason,
    })

# Numeric features actually used
for feat in NUMERIC_FEATURES:
    if feat in df.columns:
        audit_rows.append({
            'Feature': feat,
            'Decision-time available?': 'Yes',
            'Leakage risk': 'None (pre-decision metrics)',
            'Verdict': 'SAFE',
            'Reason': 'Known at prediction time: keyword data, content properties, 90-day trailing metrics',
        })

# Categorical features actually used
for feat in CATEGORICAL_FEATURES:
    if feat in df.columns:
        audit_rows.append({
            'Feature': feat,
            'Decision-time available?': 'Yes',
            'Leakage risk': 'None (derived from content properties)',
            'Verdict': 'SAFE',
            'Reason': 'Tier/bucket derived from known content properties',
        })

audit_df = pd.DataFrame(audit_rows)
print(f"Audited {len(audit_df)} features")
print(f"SAFE: {(audit_df['Verdict'] == 'SAFE').sum()}, EXCLUDE: {(audit_df['Verdict'] == 'EXCLUDE').sum()}")
audit_df

Audited 33 features
SAFE: 26, EXCLUDE: 7


,Feature,Decision-time available?,Leakage risk,Verdict,Reason
0,trend_direction,No,Label-derived,EXCLUDE,Label source: is_declining_label is derived fr...
1,trend_pct,No,Label-derived,EXCLUDE,Label source: trend_direction is computed from...
2,is_declining_label,No,Label-derived,EXCLUDE,The target variable itself
3,content_id,No,Identifier,EXCLUDE,"Pseudonymous identifier, not predictive"
4,client_id,No,Identifier,EXCLUDE,"Used for grouped splits only, never a feature"
5,provider_used,No,Decision-derived,EXCLUDE,Not available at decision time (post-publicati...
6,model_used,No,Decision-derived,EXCLUDE,Not available at decision time (post-publicati...
7,search_volume,Yes,None (pre-decision metrics),SAFE,"Known at prediction time: keyword data, conten..."
8,competition,Yes,None (pre-decision metrics),SAFE,"Known at prediction time: keyword data, conten..."
9,cpc,Yes,None (pre-decision metrics),SAFE,"Known at prediction time: keyword data, conten..."


**Leakage verdict:** No features in the final model contain label-derived or future information. The 18 numeric and 8 categorical features are all computed from data available at decision time (keyword context, content properties, trailing 90-day metrics). The 7 excluded fields are all either label-derived, identifiers, or post-publication metadata. The 90-day trailing window features do not overlap with the label window (the label is defined on a 30-day trend comparison within the trailing window, but the features are aggregates, not sub-windows that would contain the label outcome).

## 4. Real failure examples

*Use actual held-out test predictions. Show a small number of real failure examples.*

In [9]:
# Get failure examples from the Week 5 holdout test set
test_eval = test_df[['content_id', 'client_id', 'is_declining_label', 'content_type',
                     'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update',
                     'content_age_days', 'search_volume', 'competition_level']].copy()
test_eval['lr_prob'] = lr_probs
test_eval['lr_pred'] = (lr_probs >= 0.5).astype(int)

fp = test_eval[(test_eval['lr_pred'] == 1) & (test_eval['is_declining_label'] == 0)].copy()
fn = test_eval[(test_eval['lr_pred'] == 0) & (test_eval['is_declining_label'] == 1)].copy()

print(f"False positives: {len(fp):,} ({len(fp)/len(test_eval):.1%})")
print(f"False negatives: {len(fn):,} ({len(fn)/len(test_eval):.1%})")

False positives: 395 (17.0%)
False negatives: 394 (16.9%)


### False positive examples (model predicted declining, but actually stable/up)

In [10]:
# Show 3 real false positive examples
cols_to_show = ['content_id', 'content_type', 'impressions_90d', 'avg_position', 'ctr',
                'days_since_last_update', 'content_age_days', 'search_volume', 'lr_prob']
print("=== False Positive Examples ===")
print(f"Total FP: {len(fp):,}")
print("\nCharacteristic stats:")
print(f"  Avg impressions_90d: {fp['impressions_90d'].mean():,.0f}")
print(f"  Avg CTR: {fp['ctr'].mean():.2f}%")
print(f"  Avg avg_position: {fp['avg_position'].mean():.1f}")
print(f"  Content types: {fp['content_type'].value_counts().to_dict()}")
print("\n3 concrete examples:")
for i, (_, row) in enumerate(fp.head(3).iterrows()):
    print(f"  {row['content_id']}: type={row['content_type']}, imp={row['impressions_90d']:,}, pos={row['avg_position']:.1f}, ctr={row['ctr']:.2f}%, age={row['content_age_days']}d, prob={row['lr_prob']:.3f}")
    if row['content_type'] == 'keyword article':
        print(f"    Why likely wrong: moderate impressions and low CTR do not necessarily mean declining — the model interprets low engagement as decline signal")
    else:
        print(f"    Why likely wrong: feedly article with limited keyword data may trigger false engagement-based decline signal")

=== False Positive Examples ===
Total FP: 395

Characteristic stats:
  Avg impressions_90d: 3,714
  Avg CTR: 0.81%
  Avg avg_position: 13.8
  Content types: {'keyword article': 369, 'feedly article': 26}

3 concrete examples:
  content_d7cbd76b788d: type=keyword article, imp=17,992, pos=6.4, ctr=0.11%, age=144d, prob=0.704
    Why likely wrong: moderate impressions and low CTR do not necessarily mean declining — the model interprets low engagement as decline signal
  content_c3e86d4031b6: type=keyword article, imp=801, pos=10.2, ctr=0.00%, age=175d, prob=0.828
    Why likely wrong: moderate impressions and low CTR do not necessarily mean declining — the model interprets low engagement as decline signal
  content_7dff534db3ae: type=keyword article, imp=13,347, pos=5.3, ctr=0.31%, age=92d, prob=0.549
    Why likely wrong: moderate impressions and low CTR do not necessarily mean declining — the model interprets low engagement as decline signal


### False negative examples (model missed a declining page)

In [11]:
print("=== False Negative Examples ===")
print(f"Total FN: {len(fn):,}")
print("\nCharacteristic stats:")
print(f"  Avg impressions_90d: {fn['impressions_90d'].mean():,.0f}")
print(f"  Avg CTR: {fn['ctr'].mean():.2f}%")
print(f"  Avg avg_position: {fn['avg_position'].mean():.1f}")
print(f"  Content types: {fn['content_type'].value_counts().to_dict()}")
print("\n3 concrete examples:")
for i, (_, row) in enumerate(fn.head(3).iterrows()):
    print(f"  {row['content_id']}: type={row['content_type']}, imp={row['impressions_90d']:,}, pos={row['avg_position']:.1f}, ctr={row['ctr']:.2f}%, age={row['content_age_days']}d, prob={row['lr_prob']:.3f}")
    print(f"    Why likely missed: very low impressions (~{row['impressions_90d']:,}) provide too little data for the model to learn a decline pattern")

=== False Negative Examples ===
Total FN: 394

Characteristic stats:
  Avg impressions_90d: 377
  Avg CTR: 3.60%
  Avg avg_position: 14.9
  Content types: {'keyword article': 231, 'feedly article': 163}

3 concrete examples:
  content_326fa2fa449f: type=keyword article, imp=4, pos=8.3, ctr=0.00%, age=91d, prob=0.473
    Why likely missed: very low impressions (~4) provide too little data for the model to learn a decline pattern
  content_0af426466565: type=keyword article, imp=9, pos=3.6, ctr=0.00%, age=91d, prob=0.421
    Why likely missed: very low impressions (~9) provide too little data for the model to learn a decline pattern
  content_ea851c8c0ad2: type=keyword article, imp=43, pos=2.4, ctr=0.00%, age=125d, prob=0.480
    Why likely missed: very low impressions (~43) provide too little data for the model to learn a decline pattern


## 5. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claims from Week 5 to audit

| # | Original claim | Assessment | Rewritten |
|---|---|---|---|
| 1 | "The model can reliably predict which content items are declining" | Overstates: test set is a single 6-client holdout; 39.1% vs 55.5% base rate shift suggests sensitivity to which clients are held out | "On the observed 6-client holdout, the model ranked content by estimated decline probability with ROC-AUC 0.70 (logistic) and 0.75 (random forest) — directional evidence, not a reliability guarantee." |
| 2 | "Both models beat the Week 4 rule on all metrics" | Partially inaccurate: Logistic Regression lost on Precision@20 and Precision@50 | "Both models beat the baseline on ROC-AUC and Average Precision. Logistic Regression matched the baseline at P@100 and beat it at P@500. Random Forest beat the baseline at all Precision@K levels. These are measured results on the held-out clients, not a general claim across all clients." |
| 3 | "Random Forest wins on all metrics" | Overgeneralizes: this is one holdout split, not a guaranteed ordering | "Random Forest scored higher than Logistic Regression on every metric in this particular client holdout. Cross-validation confirms the direction but with a smaller gap (ROC-AUC 0.71 vs 0.71), so the magnitude of the difference is directional, not definitive." |
| 4 | "The model's strongest signals are log_impressions_90d and word_count" | Reasonable but should note this is descriptive of the linear model, not a causal claim | "In the logistic regression, the features with the largest absolute coefficients were log_impressions_90d and word_count. These are measured associations within this model, not evidence that changing word count will change decline risk." |

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.